In [ ]:
from download_tools import *
from preprocessing import *
from km_estimation import *
import matplotlib.pyplot as plt

In [ ]:
sites_southeast = ["02177000", "02178400", "03455500", "03500000", "02143500"]
sites_texas_ok = ["08167500", "08171000", "08195000", "07311500", "07332500"]
sites_mountain = ["06043500", "06191500", "09065500", "09066000", "09085000"]
sites_pnw = ["14158500", "14306500", "12189500"]
sites_newengland = ["01052500", "01055000"]

sites = sites_southeast + sites_texas_ok + sites_mountain + sites_pnw + sites_newengland

start_date = "2005-01-01"
end_date = "2026-01-01"
dt = 15

y_col = '00065' #Depth
target_folder = 'data'


def main():
    # # Step 1: Download (do once, save raw_data)
    results = {}
    raw_data = get_all_sites(sites, target_folder, start_date, end_date)
    for site_id, site_data in raw_data.items():
        if '00065' not in site_data.columns:
            print(f"Missing '00065' in {site_id}")
            print(f"Available columns: {site_data.columns.tolist()}")
            continue
        site_dictionary = create_site_dictionary(site_data, dt= dt, omega = 0.49,c=2, std_window=50)
        compute_KM(site_dictionary)
        results[site_id] = site_dictionary
    return results


In [ ]:
results = main()

In [ ]:
from Models.product import ProductModel
from Models.piecewise import LinearPowerLaw


site = sites[1]
edges = results[site]['KM_results']['edges']
moments = results[site]['KM_results']['F1']

drift_model = LinearPowerLaw()
drift_model.fit(edges, moments)
drift_model.params
y_fit = np.linspace(0,6)
drift_fit = drift_model.evaluate(y_fit)

drift_model = ProductModel(upper_trim = 0.7)
drift_model.fit(edges, moments)
drift_model.params
y_fit = np.linspace(0,6)
drift_fit = drift_model.evaluate(y_fit)



In [ ]:
from plotter import Plotter

fit = (y_fit, drift_fit)
plotter = Plotter(site, results)
plotter.plot(results, fit)

In [ ]:
for site in results.keys():
    edges = results[site]['KM_results']['edges']
    moments = results[site]['KM_results']['F1']
    
    drift_model = ProductModel(upper_trim=0.9)
    drift_model.fit(edges, moments)
    
    y_fit = np.linspace(0, edges.max(), 100)
    drift_fit = drift_model.evaluate(y_fit)
    
    results[site]['Fit_Results'] = {
        'model': drift_model,
        'model_fit': (y_fit, drift_fit)
    }

    

In [ ]:
n_sites = len(results)
ncols = 4
nrows = int(np.ceil(n_sites / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten()

for i, site in enumerate(results.keys()):
    ax = axes[i]
    
    edges = results[site]['KM_results']['edges']
    F1 = results[site]['KM_results']['F1']
    y_fit, drift_fit = results[site]['Fit_Results']['model_fit']
    
    ax.plot(edges, F1, 'k-', lw=1, alpha=0.7, label='Data')
    ax.plot(y_fit, drift_fit, 'r-', lw=1.5, label='Fit')
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    # ax.set_yscale('symlog', linthresh=1e-6)
    ax.set_title(site, fontsize=9)
    ax.set_xlabel('y')
    ax.set_ylabel('$F_1$')
    
    if i == 0:
        ax.legend(fontsize=8)

# Hide empty subplots
for i in range(n_sites, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Extract parameters from all sites
y_eqs = []
ms = []
aas = []
alphas = []

for site in results.keys():
    params = results[site]['Fit_Results']['model'].params
    
    if np.isfinite(params['y_eq']):
        y_eqs.append(params['y_eq'])
    if np.isfinite(params['m']) and params['m'] != 0:
        ms.append(params['m'])
    if np.isfinite(params['a']) and params['a'] > 0:
        aas.append(params['a'])
    if np.isfinite(params['alpha']):
        alphas.append(params['alpha'])

# Plot histograms
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# y_eq
ax = axes[0, 0]
ax.hist(y_eqs, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(y_eqs), color='r', linestyle='--', label=f'mean={np.mean(y_eqs):.2f}')
ax.axvline(np.median(y_eqs), color='orange', linestyle='--', label=f'median={np.median(y_eqs):.2f}')
ax.set_xlabel('$y_{eq}$')
ax.set_ylabel('Count')
ax.set_title('Equilibrium point')
ax.legend()

# m (log scale)
ax = axes[0, 1]
log_ms = np.log10(np.abs(ms))
ax.hist(log_ms, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(log_ms), color='r', linestyle='--', label=f'mean={np.mean(log_ms):.2f}')
ax.axvline(np.median(log_ms), color='orange', linestyle='--', label=f'median={np.median(log_ms):.2f}')
ax.set_xlabel('$\\log_{10}|m|$')
ax.set_ylabel('Count')
ax.set_title('Linear slope (log)')
ax.legend()

# a (log scale)
ax = axes[1, 0]
log_as = np.log10(aas)
ax.hist(log_as, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(log_as), color='r', linestyle='--', label=f'mean={np.mean(log_as):.2f}')
ax.axvline(np.median(log_as), color='orange', linestyle='--', label=f'median={np.median(log_as):.2f}')
ax.set_xlabel('$\\log_{10}(a)$')
ax.set_ylabel('Count')
ax.set_title('Power law coefficient (log)')
ax.legend()

# alpha
ax = axes[1, 1]
ax.hist(alphas, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(np.mean(alphas), color='r', linestyle='--', label=f'mean={np.mean(alphas):.2f}')
ax.axvline(np.median(alphas), color='orange', linestyle='--', label=f'median={np.median(alphas):.2f}')
ax.set_xlabel('$\\alpha$')
ax.set_ylabel('Count')
ax.set_title('Power law exponent')
ax.legend()

plt.tight_layout()
plt.show()

# Print summary
print("\n=== Parameter Summary ===")
print(f"y_eq:  mean={np.mean(y_eqs):.2f}, median={np.median(y_eqs):.2f}, std={np.std(y_eqs):.2f}")
print(f"m:     mean={np.mean(ms):.2e}, median={np.median(ms):.2e}")
print(f"a:     mean={np.mean(aas):.2e}, median={np.median(aas):.2e}")
print(f"alpha: mean={np.mean(alphas):.2f}, median={np.median(alphas):.2f}, std={np.std(alphas):.2f}")